In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datatable as dt

In [3]:
#Input excel file, need one tab of Results then a tab for each VESIcal model output
#This Results tab must be blank with only a column for SheetName and the below described rows for export. See example input sheet.
input_excel_path = "Timanfaya_Derivative_Input.xlsx" 

#Output excel file path
output_excel_path = "Timanfaya_Derivative_Output.xlsx"

#Reading all sheets, one for each model
all_sheets = pd.read_excel(input_excel_path, sheet_name=None)

#Exporting all the sheets
summary_rows = []

#Looping through all the sheets (specific modeled compositions and degassing paths)
for sheet_name, df in all_sheets.items():
    try:
        #Finding point of fluid saturation ("First Fluid"), this is the first point for each of the VESIcal models
        fluid_vf = df['FluidProportion_wt'].to_numpy().flatten()
        fluid_vf_diff = np.diff(fluid_vf)
        fluid_vf_diff = np.concatenate([[np.nan], fluid_vf_diff])
        df['Fluid_VF_diff'] = fluid_vf_diff

        sign_change_indices = np.where(np.diff(np.sign(fluid_vf_diff)) != 0)[0]

        #Finding the branch start of XH2O degassing at a 70% threshold (note, this 70% threshold works for VESIcal models with 50 bar steps, 
        # but can/will be different for other MagmaSat platforms which use either closer or further spaced dP steps
        x = df['Pressure_bars'].values
        y = df['XH2O_fl'].values
        dy_dx = np.gradient(y, x)
        d2y_dx2 = np.gradient(dy_dx, x)
        threshold = np.percentile(np.abs(d2y_dx2), 70)
        branch_indices = np.where(np.abs(d2y_dx2) > threshold)[0]

        if len(branch_indices) == 0:
            continue  

        branch_start = branch_indices[0]
        branch_end = branch_indices[-1]

        start_Pressure_bars = x[branch_start]
        start_X_H2O = df.loc[branch_start, 'XH2O_fl']
        start_X_CO2 = df.loc[branch_start, 'XCO2_fl']
        start_H2O_liq = df.loc[branch_start, 'H2O_liq']
        start_CO2_liq = df.loc[branch_start, 'CO2_liq']
        start_fluidpro = df.loc[branch_start, 'FluidProportion_wt']

        #Exporting Rows to Save back to the results sheet
        summary_data = {
            'SheetName': sheet_name,
            'First_Fluid_Index': sign_change_indices[0] if len(sign_change_indices) > 0 else np.nan,
            'First_Fluid_Pressure_bars': df.loc[sign_change_indices[0], 'Pressure_bars'] if len(sign_change_indices) > 0 else np.nan,
            'First_Fluid_FluidProportion_wt': df.loc[sign_change_indices[0], 'FluidProportion_wt'] if len(sign_change_indices) > 0 else np.nan,
            'First_Fluid_H2O_liq': df.loc[sign_change_indices[0], 'H2O_liq'] if len(sign_change_indices) > 0 else np.nan,
            'First_Fluid_CO2_liq': df.loc[sign_change_indices[0], 'CO2_liq'] if len(sign_change_indices) > 0 else np.nan,
            'First_Fluid_XH2O_fl': df.loc[sign_change_indices[0], 'XH2O_fl'] if len(sign_change_indices) > 0 else np.nan,
            'First_Fluid_XCO2_fl': df.loc[sign_change_indices[0], 'XCO2_fl'] if len(sign_change_indices) > 0 else np.nan,
            'BranchStart_Index': branch_start,
            'BranchStart_Pressure_bars': start_Pressure_bars,
            'BranchStart_FluidProportion_wt': start_fluidpro,
            'BranchStart_H2O_liq': start_H2O_liq,
            'BranchStart_CO2_liq': start_CO2_liq,
            'BranchStart_XH2O_fl': start_X_H2O,
            'BranchStart_XCO2_fl': start_X_CO2
        }

        summary_rows.append(summary_data)

    except Exception: 
        pass

#Translating back to a dataframe for export
summary_df = pd.DataFrame(summary_rows)

#Save to output excel file, which was named at the top of this code block
summary_df.to_excel(output_excel_path, index=False)
print(f"results saved to '{output_excel_path}'")


results saved to 'Timanfaya_Derivative_Output.xlsx'
